In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 20000

data = {
    'user_id': range(1, n+1),
    'group': np.random.choice(['A', 'B'], n),
    'device': np.random.choice(['Mobile', 'Desktop', 'Tablet'], n, p=[0.6, 0.3, 0.1]),
    'country': np.random.choice(['UAE', 'Saudi', 'Egypt', 'UK'], n, p=[0.4, 0.2, 0.2, 0.2]),
    'visited': 1
}

df = pd.DataFrame(data)
df['purchased'] = np.where((df['group'] == 'A') & (np.random.rand(n) < 0.021), 1, 0)
df['purchased'] = np.where((df['group'] == 'B') & (np.random.rand(n) < 0.026), 1, df['purchased'])
df['revenue'] = np.where(df['purchased'] == 1, np.random.uniform(50, 500, n), 0)

df.to_csv('ab_test_data.csv', index=False)
print("Done! File 'ab_test_data.csv' created")
df.head(10)

Done! File 'ab_test_data.csv' created


,user_id,group,device,country,visited,purchased,revenue
0,1,A,Mobile,Egypt,1,0,0.0
1,2,B,Mobile,Saudi,1,0,0.0
2,3,A,Mobile,UK,1,0,0.0
3,4,A,Desktop,UAE,1,0,0.0
4,5,A,Mobile,Saudi,1,0,0.0
5,6,B,Desktop,Egypt,1,0,0.0
6,7,A,Mobile,UK,1,0,0.0
7,8,A,Desktop,Saudi,1,0,0.0
8,9,A,Desktop,Egypt,1,0,0.0
9,10,B,Desktop,UAE,1,0,0.0


In [2]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv('ab_test_data.csv')

conversion = df.groupby('group')['purchased'].agg(['sum', 'count'])
conversion['conversion_rate'] = conversion['sum'] / conversion['count'] * 100
print(conversion)

purchases_A = conversion.loc['A', 'sum']
users_A = conversion.loc['A', 'count']
purchases_B = conversion.loc['B', 'sum'] 
users_B = conversion.loc['B', 'count']

z_stat, p_value = proportions_ztest([purchases_A, purchases_B], [users_A, users_B])
print(z_stat)
print(p_value)

if p_value < 0.05:
    lift = (conversion.loc['B', 'conversion_rate'] - conversion.loc['A', 'conversion_rate'])
    print(lift)
    print("Launch B")
else:
    print("Keep A")

       sum  count  conversion_rate
group                             
A      211  10020         2.105788
B      242   9980         2.424850
-1.5163486529462273
0.12943121532330631
Keep A


In [3]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv('ab_test_data.csv')

table = df.groupby('group')['purchased'].agg(['sum', 'count'])
table['conversion_rate'] = table['sum'] / table['count'] * 100
print(table)

purchases_A = table.loc['A', 'sum']
users_A = table.loc['A', 'count']
purchases_B = table.loc['B', 'sum']
users_B = table.loc['B', 'count']

z_stat, p_value = proportions_ztest([purchases_A, purchases_B], [users_A, users_B])
print(z_stat)
print(p_value)

if p_value < 0.05:
    lift = table.loc['B', 'conversion_rate'] - table.loc['A', 'conversion_rate']
    print(lift)
    print("Launch B")
else:
    print("Keep A")

       sum  count  conversion_rate
group                             
A      211  10020         2.105788
B      242   9980         2.424850
-1.5163486529462273
0.12943121532330631
Keep A


In [4]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest


data = {
    'group': ['A']*10000 + ['B']*10000,
    'purchased': [0]*9790 + [1]*210 + [0]*9740 + [1]*260 
}
df = pd.DataFrame(data)
table = df.groupby('group')['purchased'].agg(['sum', 'count'])
table['conversion_rate'] = table['sum'] / table['count'] * 100
print("aggregation table")
print(table)

purchases_A = table.loc['A', 'sum']
users_A = table.loc['A', 'count']
purchases_B = table.loc['B', 'sum']
users_B = table.loc['B', 'count']

z_stat, p_value = proportions_ztest([purchases_A, purchases_B], [users_A, users_B])
print(f"\np-value = {p_value:.4f}")

if p_value < 0.05:
    lift = table.loc['B', 'conversion_rate'] - table.loc['A', 'conversion_rate']
    print(f"Lift = {lift:.2f}%")
    print("Launch B")
else:
    print("Keep A")

aggregation table
       sum  count  conversion_rate
group                             
A      210  10000              2.1
B      260  10000              2.6

p-value = 0.0196
Lift = 0.50%
Launch B


In [5]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

data = {
    'group': ['A']*500 + ['B']*500,
    'purchased': [0]*480 + [1]*20 + [0]*470 + [1]*30 # A=20, B=30
}
df = pd.DataFrame(data)

table = df.groupby('group')['purchased'].agg(['sum', 'count'])
table['conversion_rate'] = table['sum'] / table['count'] * 100
print(table)

purchases_A = table.loc['A', 'sum']
users_A = table.loc['A', 'count']
purchases_B = table.loc['B', 'sum']
users_B = table.loc['B', 'count']

z_stat, p_value = proportions_ztest([purchases_A, purchases_B], [users_A, users_B])
print(f"p-value = {p_value:.4f}")

if p_value < 0.05:
    lift = table.loc['B', 'conversion_rate'] - table.loc['A', 'conversion_rate']
    print(f"Lift = {lift:.2f}%")
    print("Launch B")
else:
    print("Keep A")

       sum  count  conversion_rate
group                             
A       20    500              4.0
B       30    500              6.0
p-value = 0.1468
Keep A


In [7]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

data = {
    'group': ['A']*10000 + ['B']*10000,
    'purchased': [0]*9700 + [1]*300 + [0]*9750 + [1]*250 # A=300, B=250
}
df = pd.DataFrame(data)

table = df.groupby('group')['purchased'].agg(['sum', 'count'])
table['conversion_rate'] = table['sum'] / table['count'] * 100
print(table)

purchases_A = table.loc['A', 'sum']
users_A = table.loc['A', 'count']
purchases_B = table.loc['B', 'sum']
users_B = table.loc['B', 'count']

z_stat, p_value = proportions_ztest([purchases_A, purchases_B], [users_A, users_B])
print(f"p-value = {p_value:.4f}")

if p_value < 0.05:
    lift = table.loc['B', 'conversion_rate'] - table.loc['A', 'conversion_rate']
    print(f"Lift = {lift:.2f}%")
    if lift >0:
        print("Launch B")
    else:
        print("Keep A - B is worse")
else:
    print("Keep A - Not significant")

       sum  count  conversion_rate
group                             
A      300  10000              3.0
B      250  10000              2.5
p-value = 0.0306
Lift = -0.50%
Keep A - B is worse
